# Foundation Model – Data Pipeline Validation

**Purpose:** Validate the encoded and sequenced Parquet outputs generated by the training data pipeline.

This notebook is designed for trainees. You do not need to change the original pipeline code. Update the paths in **Section 2**, check the column settings in **Section 6**, then run every cell from top to bottom (**Run → Run All Cells**) and review the report in **Section 13**.

**The two outputs**
- **Encoded output**: one row per event, holding that event's token IDs (for example `key_ids` and `value_ids` lists).
- **Sequenced output**: one row per user, holding the user's whole history as flat token lists. An offsets column (for example `event_offsets`) marks where each event starts, and per-event lists hold the timestamps and calendar features.

**What this notebook checks**
- Parquet files can be found and read, and every file of one output has the same schema
- Schema, columns and data types
- Empty data, nulls, duplicate rows and duplicate users
- Token IDs: whole numbers, inside the vocabulary, no `[PAD]` / `[MASK]`, a low `[UNK]` share
- Sequence structure: token lists of equal length, offsets that match the tokens, one value per event in every per-event list
- Timestamps decode to real dates and are in time order inside each sequence
- The encoded and sequenced outputs hold the same users, events and tokens
- A final PASS / WARNING / FAIL report

> The column names in Section 6 are pre-filled with the names used by the reference tokenizer output. If your pipeline uses different names, change them there after looking at the schema in Section 5. A check whose setting is `None` is reported as SKIPPED rather than guessed.

## 1. Import dependencies and set up the report

Run this cell once in the environment where you will execute the notebook. If a package is missing, uncomment the `%pip install` line, run it, then restart the kernel.

Every check writes one row into a shared results table, which Section 13 turns into the final report. If a check crashes, it is recorded as **ERROR** and the notebook carries on, so **Run All** always reaches the report. Re-running a cell replaces that cell's earlier results instead of adding duplicates.

In [ ]:
# If required, run:
# %pip install pandas pyarrow numpy

import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_colwidth", 100)
print(f"pandas {pd.__version__} | numpy {np.__version__}")


# one row per check, keyed so that re-running a cell replaces its own rows
results = {}
current_section = None


def start_section(name):
    # forget the rows this section wrote on an earlier run
    global current_section
    current_section = name
    for key in [k for k in results if k[0] == name]:
        del results[key]


def record_check(dataset, check, status, details):
    results[(current_section, dataset, check)] = {
        "section": current_section,
        "dataset": dataset,
        "check":   check,
        "status":  status,
        "details": str(details),
    }


def run_safely(dataset, step, fn, *args):
    # a crash inside one check becomes an ERROR row instead of stopping Run All
    try:
        fn(*args)
    except Exception as exc:
        record_check(dataset, step, "ERROR", f"{type(exc).__name__}: {exc}")
        print(f"[{dataset}] {step} stopped with an error: {type(exc).__name__}: {exc}")


def show_section_results():
    rows = [r for r in results.values() if r["section"] == current_section]
    if not rows:
        print("No checks recorded in this section.")
        return
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(pd.DataFrame(rows).drop(columns="section"))


def is_list_value(value):
    return isinstance(value, (list, tuple, np.ndarray))


def list_length(value):
    # -1 marks a cell that is not a list (null or a single value)
    return len(value) if is_list_value(value) else -1


def flatten_values(series):
    # join every list cell (or single value) of a column into one numpy array
    parts = []
    for item in series.dropna():
        if is_list_value(item):
            parts.append(np.asarray(item))
        else:
            parts.append(np.asarray([item]))
    if not parts:
        return np.array([])
    return np.concatenate(parts)


def example_user(df, mask, user_col):
    # name the first offending row, so the problem can be found quickly
    if not mask.any():
        return ""
    row = int(np.flatnonzero(np.asarray(mask))[0])
    if user_col and user_col in df.columns:
        return f" First: {user_col}={df[user_col].iloc[row]} (row {row})."
    return f" First: row {row}."

## 2. Set your local paths

You can provide either:
- A **single `.parquet` file**, or
- A **folder** containing one or more Parquet files (subfolders are searched too).

Use a raw string (`r"..."`) for Windows paths to avoid backslash issues. Set a path to `None` if you do not have that output yet.

`VOCAB_FILE` is optional but recommended: point it at the `tokenizer.json` (or `shard_manifest.json`) written by the tokenize step. Its `vocab_size` sets the upper limit for token IDs, and `tokenizer.json` lets Section 12 show key and value names instead of bare numbers.

In [ ]:
# CHANGE THESE PATHS

ENCODED_PATH   = r"C:\path\to\your\encoded_output"
SEQUENCED_PATH = r"C:\path\to\your\sequenced_output"

# Optional: tokenizer.json or shard_manifest.json from the tokenize step.
# Example:
# VOCAB_FILE = r"C:\path\to\your\tokenized_output\tokenizer\tokenizer.json"
VOCAB_FILE = None

# Optional: read at most this many rows per output (None = read everything).
# Use it only when an output does not fit in memory; the cross-check in
# Section 11 is skipped when rows are capped.
MAX_ROWS_PER_OUTPUT = None

## 3. Find the Parquet files

This cell checks whether the paths exist, lists the Parquet files found with their row counts, and checks that every file of one output has the same schema. Different schemas inside one output usually mean files from different pipeline runs are mixed in the same folder.

In [ ]:
start_section("3. Files")


def find_parquet_files(path):
    if path is None or str(path).strip() == "":
        return []

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Path does not exist: {path}\nUpdate the paths in Section 2.")

    if path.is_file():
        if path.suffix.lower() != ".parquet":
            raise ValueError(f"Expected a .parquet file, received: {path}")
        return [path]

    # rglob rather than glob, so folder names containing [ ] are not read as patterns;
    # names starting with . or _ are hidden or marker files, not data
    files = [p for p in path.rglob("*.parquet") if not p.name.startswith((".", "_"))]
    return sorted(files)


def check_files(files, path, label):
    if path is None or str(path).strip() == "":
        record_check(label, "Output provided", "SKIPPED", "No path set in Section 2.")
        return
    if not files:
        record_check(label, "Output provided", "FAIL", f"No .parquet files under {path}.")
        return

    # group files by schema; only column names and types count, not metadata
    groups = {}
    for f in files:
        schema = pq.read_schema(f).remove_metadata()
        groups.setdefault(schema.to_string(), []).append(f.name)

    if len(groups) == 1:
        record_check(label, "Same schema in every file", "PASS",
                     f"{len(files)} file(s), one schema.")
    else:
        kinds = "; ".join(f"{len(names)} file(s) like {names[0]}" for names in groups.values())
        record_check(label, "Same schema in every file", "WARNING",
                     f"{len(groups)} different schemas: {kinds}. Columns missing from "
                     "some files will show up as nulls after loading.")


encoded_files   = find_parquet_files(ENCODED_PATH)
sequenced_files = find_parquet_files(SEQUENCED_PATH)

print(f"Encoded Parquet files  : {len(encoded_files)}")
print(f"Sequenced Parquet files: {len(sequenced_files)}")

if not encoded_files and not sequenced_files:
    raise ValueError("No Parquet files found. Please check the paths in Section 2.")

run_safely("Encoded",   "Files", check_files, encoded_files, ENCODED_PATH, "Encoded")
run_safely("Sequenced", "Files", check_files, sequenced_files, SEQUENCED_PATH, "Sequenced")

all_files = encoded_files + sequenced_files
display(pd.DataFrame({
    "output_type": ["encoded"] * len(encoded_files) + ["sequenced"] * len(sequenced_files),
    "file_path":   [str(f) for f in all_files],
    "rows":        [pq.read_metadata(f).num_rows for f in all_files],
    "size_mb":     [round(f.stat().st_size / 1e6, 2) for f in all_files],
}))
show_section_results()

## 4. Read the Parquet outputs

The files are combined into one DataFrame per output type. This is convenient for validation and exploration.

For very large outputs, loading every row into memory may not be practical. In that case set `MAX_ROWS_PER_OUTPUT` in Section 2 to validate the first rows only.

In [ ]:
def load_parquet_files(files, max_rows=None):
    if not files:
        return None, False

    frames    = []
    rows_read = 0
    for f in files:
        if max_rows is not None and rows_read >= max_rows:
            break
        table = pq.read_table(str(f))
        if max_rows is not None:
            table = table.slice(0, max_rows - rows_read)
        frames.append(table.to_pandas())
        rows_read += table.num_rows

    total_rows = sum(pq.read_metadata(f).num_rows for f in files)
    df = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
    return df, rows_read < total_rows


encoded_df,   encoded_capped   = load_parquet_files(encoded_files, MAX_ROWS_PER_OUTPUT)
sequenced_df, sequenced_capped = load_parquet_files(sequenced_files, MAX_ROWS_PER_OUTPUT)

for label, df, capped in [("ENCODED", encoded_df, encoded_capped),
                          ("SEQUENCED", sequenced_df, sequenced_capped)]:
    if df is None:
        continue
    note = "  (capped by MAX_ROWS_PER_OUTPUT)" if capped else ""
    print(f"{label} OUTPUT")
    print(f"Rows: {len(df):,} | Columns: {len(df.columns)}{note}\n")

## 5. Inspect the schema

Before applying checks, understand what your pipeline actually produced.

Review:
- Column names and data types
- Whether each column holds a single value or a list per row (`holds`)
- A few sample rows

**Do not assume that a column named `key_ids` or `value_ids` exists.** Your pipeline version may use different names; Section 6 is where you set them.

In [ ]:
def value_kind(series):
    # what one cell holds, judged from the first non-null value
    sample = series.dropna()
    if len(sample) == 0:
        return "all null"
    first = sample.iloc[0]
    if is_list_value(first):
        return f"list ({len(first)} items in first row)"
    if isinstance(first, dict):
        return "struct"
    return "single value"


def show_schema(df, label):
    if df is None:
        print(f"{label}: not provided")
        return

    print(f"\n--- {label} schema ---")
    schema = pd.DataFrame({
        "column":     df.columns,
        "data_type":  [str(dtype) for dtype in df.dtypes],
        "holds":      [value_kind(df[col]) for col in df.columns],
        "null_count": [int(df[col].isna().sum()) for col in df.columns],
    })
    display(schema)
    print("Sample rows:")
    display(df.head(5))


show_schema(encoded_df, "ENCODED")
show_schema(sequenced_df, "SEQUENCED")

## 6. Configure the columns to validate

After inspecting Section 5, make sure the column names below match your output. The sequenced defaults are the names used by the reference tokenizer output.

| Setting | What it means |
|---|---|
| `user_column` | Column holding the user / customer ID |
| `timestamp_column` | Event time: a single value per row (encoded) or a list per row (sequenced) |
| `token_columns` | Columns holding token IDs. The first two are read as key IDs and value IDs in Section 12 |
| `aligned_columns` | List columns that must have the **same length on every row** (one entry per token) |
| `length_column` | A stored token count to compare with the real list length |
| `offsets_column` | Where each event starts inside the token lists. Must start at 0, never go down, and end at the token count |
| `per_event_columns` | List columns with **one value per event** (timestamps, calendar fields, source) |
| `profile_*` | The same three settings for profile tokens stored next to the event tokens |
| `unique_users` | `True` when every user must appear in exactly one row |
| `range_columns` | Allowed `(min, max)` for numeric columns. `None` means no limit on that side |

Set any setting to `None` (or an empty list / dict) to skip that check. Any setting can be used in either output.

In [ ]:
# Update these values based on the schema shown in Section 5.

# encoded output: one row per event
ENCODED = {
    "user_column":       "user_id",
    "timestamp_column":  None,                  # Example: "event_ts" or "ts"
    "token_columns":     ["key_ids", "value_ids"],
    "aligned_columns":   ["key_ids", "value_ids"],
    "length_column":     None,                  # Example: "n_tokens"
    "offsets_column":    None,
    "per_event_columns": [],
    "unique_users":      False,
    "range_columns":     {},
}

# sequenced output: one row per user (names used by the reference tokenizer)
SEQUENCED = {
    "user_column":       "user_id",
    "timestamp_column":  "event_ts",
    "token_columns":     ["key_ids", "value_ids", "prof_key_ids", "prof_value_ids"],
    "aligned_columns":   ["key_ids", "value_ids", "positions"],
    "length_column":     None,
    "offsets_column":    "event_offsets",
    "per_event_columns": ["event_ts", "time_log", "hour", "dow", "dom", "source_ids"],

    "profile_aligned_columns":  ["prof_key_ids", "prof_value_ids", "prof_positions"],
    "profile_offsets_column":   "prof_offsets",
    "profile_per_item_columns": ["prof_time_log", "prof_ts"],

    "unique_users":      True,
    "range_columns":     {"hour": (0, 23), "dow": (0, 6), "dom": (1, 31),
                          "time_log": (0, None), "prof_time_log": (0, None)},
}

# token ID limits, shared by both outputs
MIN_TOKEN_ID = 0
MAX_TOKEN_ID = None        # None: vocab_size - 1 from VOCAB_FILE, when it is set

# Special tokens of the reference tokenizer. [PAD] and [MASK] are added later, at
# batching and pre-training time, so they must never appear in pipeline output.
# A high [UNK] share means the tokenizer did not know many keys or values.
# Set an ID to None to skip its check.
PAD_ID        = 0
MASK_ID       = 1
UNK_ID        = 2
MAX_UNK_SHARE = 0.01       # warn when more than 1% of tokens are [UNK]

# Plausible time window. Dates outside it usually mean the wrong unit,
# for example seconds read as microseconds.
EARLIEST_TIMESTAMP = "2000-01-01"
LATEST_TIMESTAMP   = None  # None: one year from today


# read the vocabulary size (and the names in tokenizer.json) when a file is given
vocab_info = {}
if VOCAB_FILE:
    with open(VOCAB_FILE, encoding="utf-8") as fh:
        vocab_info = json.load(fh)
    print(f"Vocabulary file: vocab_size = {vocab_info.get('vocab_size')}")

if MAX_TOKEN_ID is None and vocab_info.get("vocab_size"):
    MAX_TOKEN_ID = int(vocab_info["vocab_size"]) - 1

if MAX_TOKEN_ID is None:
    print(f"Token IDs checked: >= {MIN_TOKEN_ID} (no upper limit; set VOCAB_FILE or MAX_TOKEN_ID)")
else:
    print(f"Token IDs checked: {MIN_TOKEN_ID} to {MAX_TOKEN_ID}")

## 7. Basic data-quality checks

These checks apply to both outputs:
- The output exists and is not empty
- No completely empty rows
- Null counts by column
- Duplicate full rows (list columns are compared by their contents)
- Each user appears in only one row, when `unique_users` is `True`

A duplicate full row is a warning, not automatically an error: repeated events may be valid in some datasets. A user appearing twice in the sequenced output is an error, because their history would be split across two training sequences.

In [ ]:
start_section("7. Basic")


def to_hashable(value):
    # list cells arrive as numpy arrays, which pandas cannot hash; compare them by content
    if isinstance(value, np.ndarray):
        if value.dtype != object:
            return (str(value.dtype), value.shape, value.tobytes())
        return tuple(to_hashable(v) for v in value)
    if isinstance(value, (list, tuple)):
        return tuple(to_hashable(v) for v in value)
    if isinstance(value, dict):
        return tuple((k, to_hashable(v)) for k, v in value.items())
    return value


def basic_checks(df, label, config):
    if df is None:
        record_check(label, "Dataset is not empty", "SKIPPED", "Output not provided.")
        return

    if len(df) == 0:
        record_check(label, "Dataset is not empty", "FAIL", "0 rows found.")
        return
    record_check(label, "Dataset is not empty", "PASS", f"{len(df):,} rows found.")

    # completely empty rows
    empty_rows = int(df.isna().all(axis=1).sum())
    record_check(label, "Completely empty rows",
                 "PASS" if empty_rows == 0 else "WARNING",
                 f"{empty_rows:,} completely empty rows.")

    # nulls, naming the columns that have them
    null_counts = df.isna().sum()
    null_counts = null_counts[null_counts > 0]
    if null_counts.empty:
        record_check(label, "Null values", "PASS", "0 null cells across all columns.")
    else:
        named = ", ".join(f"{col} ({n:,})" for col, n in null_counts.items())
        record_check(label, "Null values", "WARNING", f"Nulls in: {named}.")

    # duplicate full rows
    hashable_df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            hashable_df[col] = df[col].map(to_hashable)
    duplicate_count = int(hashable_df.duplicated().sum())
    record_check(label, "Duplicate full rows",
                 "PASS" if duplicate_count == 0 else "WARNING",
                 f"{duplicate_count:,} duplicate rows.")

    # one row per user
    user_col = config.get("user_column")
    if not config.get("unique_users") or not user_col:
        return
    if user_col not in df.columns:
        record_check(label, "One row per user", "FAIL",
                     f"Column '{user_col}' was not found. Update Section 6.")
        return
    repeated = df[user_col].value_counts()
    repeated = repeated[repeated > 1]
    details  = f"{df[user_col].nunique():,} users; {len(repeated):,} appear in more than one row."
    if not repeated.empty:
        details += f" First: {repeated.index[0]} ({repeated.iloc[0]} rows)."
    record_check(label, "One row per user", "PASS" if repeated.empty else "FAIL", details)


run_safely("Encoded",   "Basic checks", basic_checks, encoded_df, "Encoded", ENCODED)
run_safely("Sequenced", "Basic checks", basic_checks, sequenced_df, "Sequenced", SEQUENCED)
show_section_results()

## 8. Token ID checks

For every column in `token_columns` the notebook checks:
- The column exists and holds whole numbers (no text, no decimals, no nulls inside the lists)
- Token IDs are not below `MIN_TOKEN_ID` and not above `MAX_TOKEN_ID`. An ID above the vocabulary crashes the embedding lookup at training time
- `[PAD]` and `[MASK]` do not appear
- The `[UNK]` share stays below `MAX_UNK_SHARE`

If a token column contains lists/arrays per row, the values inside the lists are checked.

In [ ]:
start_section("8. Tokens")


def validate_token_column(df, column, label):
    if column not in df.columns:
        record_check(label, f"{column}: column exists", "FAIL",
                     "Configured column was not found. Update Section 6.")
        return

    raw_values = flatten_values(df[column])
    if len(raw_values) == 0:
        record_check(label, f"{column}: has tokens", "WARNING", "No token values found.")
        return

    # whole numbers only
    numeric     = pd.to_numeric(pd.Series(raw_values), errors="coerce")
    not_numeric = int(numeric.isna().sum())
    valid       = numeric.dropna()
    not_whole   = int((valid % 1 != 0).sum())
    record_check(label, f"{column}: whole-number IDs",
                 "PASS" if not_numeric == 0 and not_whole == 0 else "FAIL",
                 f"{len(numeric):,} values; {not_numeric:,} not numbers or null; "
                 f"{not_whole:,} with decimals.")
    if len(valid) == 0:
        return

    # lower and upper limits
    below_min = int((valid < MIN_TOKEN_ID).sum())
    above_max = int((valid > MAX_TOKEN_ID).sum()) if MAX_TOKEN_ID is not None else 0
    allowed   = f"{MIN_TOKEN_ID} to {MAX_TOKEN_ID}" if MAX_TOKEN_ID is not None else f">= {MIN_TOKEN_ID}"
    record_check(label, f"{column}: token ID range",
                 "PASS" if below_min == 0 and above_max == 0 else "FAIL",
                 f"Allowed {allowed}; observed {valid.min():.0f} to {valid.max():.0f}; "
                 f"{below_min:,} below, {above_max:,} above.")
    if MAX_TOKEN_ID is None:
        record_check(label, f"{column}: IDs inside vocabulary", "SKIPPED",
                     "No upper limit: set VOCAB_FILE (Section 2) or MAX_TOKEN_ID (Section 6).")

    # [PAD] and [MASK] belong to batching and pre-training, never to pipeline output
    for name, token_id in [("[PAD]", PAD_ID), ("[MASK]", MASK_ID)]:
        if token_id is None:
            continue
        count = int((valid == token_id).sum())
        record_check(label, f"{column}: no {name} tokens",
                     "PASS" if count == 0 else "FAIL",
                     f"{count:,} {name} tokens (ID {token_id}).")

    # a high [UNK] share means the tokenizer did not know many keys or values
    if UNK_ID is not None:
        unk_share = float((valid == UNK_ID).mean())
        record_check(label, f"{column}: [UNK] share",
                     "PASS" if unk_share <= MAX_UNK_SHARE else "WARNING",
                     f"{unk_share:.2%} of tokens are [UNK] (ID {UNK_ID}); limit {MAX_UNK_SHARE:.0%}.")


for label, df, config in [("Encoded", encoded_df, ENCODED), ("Sequenced", sequenced_df, SEQUENCED)]:
    if df is None:
        continue
    columns = config.get("token_columns") or []
    if not columns:
        record_check(label, "Token columns", "SKIPPED", "No token_columns set in Section 6.")
    for col in columns:
        run_safely(label, f"{col}: token checks", validate_token_column, df, col, label)

show_section_results()

## 9. Sequence structure checks

These checks catch the mistakes that most often break training without any error message:
- **Aligned columns** (`aligned_columns`) have the same length on every row, for example one `value_id` and one `position` for every `key_id`
- Rows are not empty
- The stored length (`length_column`) matches the real list length
- **Offsets** (`offsets_column`) start at 0, never go down, end exactly at the number of tokens, and have one entry more than the number of events
- **Per-event columns** (`per_event_columns`) have exactly one value per event
- No event has zero tokens

The same checks run on the profile tokens when the `profile_*` settings are filled in. A summary of tokens and events per row is shown so you can spot unexpected lengths.

In [ ]:
start_section("9. Structure")


def lengths_line_up(df, label, part, columns, what, user_col):
    # every column in `columns` must hold a list, all of the same length on each row
    lengths    = pd.DataFrame({col: df[col].map(list_length) for col in columns})
    not_list   = (lengths < 0).any(axis=1)
    mismatched = (lengths.nunique(axis=1) > 1) & ~not_list
    bad        = not_list | mismatched
    record_check(label, f"{part}: {what} line up",
                 "PASS" if not bad.any() else "FAIL",
                 f"{columns}: {int(mismatched.sum()):,} rows with different lengths, "
                 f"{int(not_list.sum()):,} rows that are not lists." + example_user(df, bad, user_col))
    return lengths[columns[0]]


def validate_structure(df, label, part, aligned, offsets_col, per_item, length_col, user_col):
    needed  = list(aligned) + list(per_item) + [c for c in [offsets_col, length_col] if c]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        record_check(label, f"{part}: columns exist", "FAIL", f"Not found: {missing}. Update Section 6.")
        return

    # aligned token columns
    n_tokens = lengths_line_up(df, label, part, aligned, "token lists", user_col)

    empty = n_tokens == 0
    record_check(label, f"{part}: rows are not empty",
                 "PASS" if not empty.any() else "FAIL",
                 f"{int(empty.sum()):,} empty rows out of {len(df):,}." + example_user(df, empty, user_col))

    # stored length against the real one
    if length_col:
        stored   = pd.to_numeric(df[length_col], errors="coerce")
        mismatch = stored != n_tokens
        record_check(label, f"{part}: stored length matches",
                     "PASS" if not mismatch.any() else "FAIL",
                     f"{int(mismatch.sum()):,} mismatches between '{length_col}' and the real length."
                     + example_user(df, mismatch, user_col))

    # one value per event (or profile item) in every per-item list
    n_items = None
    if per_item:
        n_items = lengths_line_up(df, label, part, per_item, "per-event lists", user_col)

    # offsets: start at 0, never go down, end at the token count, events + 1 entries
    if offsets_col:
        problems = {"missing or empty": 0, "first offset is not 0": 0, "offsets go down": 0,
                    "last offset is not the token count": 0, "count is not events + 1": 0}
        bad_rows     = np.zeros(len(df), dtype=bool)
        empty_events = 0
        for i, offsets in enumerate(df[offsets_col]):
            if not is_list_value(offsets) or len(offsets) == 0:
                problems["missing or empty"] += 1
                bad_rows[i] = True
                continue
            offsets = np.asarray(offsets, dtype=np.int64)
            steps   = np.diff(offsets)
            row_problems = [
                ("first offset is not 0",              offsets[0] != 0),
                ("offsets go down",                    (steps < 0).any()),
                ("last offset is not the token count", offsets[-1] != n_tokens.iat[i]),
                ("count is not events + 1",            n_items is not None and n_items.iat[i] >= 0
                                                       and len(offsets) != n_items.iat[i] + 1),
            ]
            for name, failed in row_problems:
                if failed:
                    problems[name] += 1
                    bad_rows[i] = True
            empty_events += int((steps == 0).sum())

        found = ", ".join(f"{name}: {count:,}" for name, count in problems.items() if count)
        record_check(label, f"{part}: offsets match the tokens",
                     "PASS" if not bad_rows.any() else "FAIL",
                     (f"Rows with problems - {found}." if found else f"All {len(df):,} rows consistent.")
                     + example_user(df, bad_rows, user_col))
        record_check(label, f"{part}: no event without tokens",
                     "PASS" if empty_events == 0 else "WARNING",
                     f"{empty_events:,} events with zero tokens.")

        if n_items is None:
            n_items = df[offsets_col].map(list_length) - 1

    # length summary for a human look
    summary = {"tokens per row": n_tokens}
    if n_items is not None:
        summary["events (items) per row"] = n_items
    print(f"\n{label} - {part}: length summary")
    display(pd.DataFrame(summary).describe().round(1))


for label, df, config in [("Encoded", encoded_df, ENCODED), ("Sequenced", sequenced_df, SEQUENCED)]:
    if df is None:
        continue
    user_col = config.get("user_column")

    if config.get("aligned_columns"):
        run_safely(label, "tokens: structure", validate_structure, df, label, "tokens",
                   config["aligned_columns"], config.get("offsets_column"),
                   config.get("per_event_columns") or [], config.get("length_column"), user_col)
    else:
        record_check(label, "tokens: structure", "SKIPPED", "No aligned_columns set in Section 6.")

    if config.get("profile_aligned_columns"):
        run_safely(label, "profile: structure", validate_structure, df, label, "profile",
                   config["profile_aligned_columns"], config.get("profile_offsets_column"),
                   config.get("profile_per_item_columns") or [], None, user_col)

show_section_results()

## 10. Timestamp and value-range checks

For the configured `timestamp_column` (a single value per row, or a list per row) the notebook:
- converts the values to dates. Numbers are read as time since 1970, and the unit (seconds, milliseconds, microseconds or nanoseconds) is worked out from their size and shown in the details;
- checks that every date falls between `EARLIEST_TIMESTAMP` and `LATEST_TIMESTAMP`. Dates in 1970 or far in the future almost always mean a unit mix-up;
- checks time order. Inside a sequence the events **must** be in time order (FAIL otherwise). In the encoded output, rows of one user out of time order give only a WARNING, since a later step may sort them.

It also checks every column in `range_columns`, for example hour of day between 0 and 23.

In [ ]:
start_section("10. Time")

earliest = pd.Timestamp(EARLIEST_TIMESTAMP, tz="UTC")
if LATEST_TIMESTAMP:
    latest = pd.Timestamp(LATEST_TIMESTAMP, tz="UTC")
else:
    latest = pd.Timestamp.now(tz="UTC") + pd.DateOffset(years=1)


def to_dates(values):
    # turn raw timestamp values into UTC dates; returns (dates, how they were read)
    series = pd.Series(values)
    if pd.api.types.is_datetime64_any_dtype(series):
        return pd.to_datetime(series, utc=True), "date/time values"

    numeric = pd.to_numeric(series, errors="coerce")
    if numeric.notna().mean() >= 0.5:
        # numbers since 1970: the size of a typical value gives the unit
        typical = numeric.abs().median()
        if typical < 1e11:
            unit = "s"
        elif typical < 1e14:
            unit = "ms"
        elif typical < 1e17:
            unit = "us"
        else:
            unit = "ns"
        dates = pd.to_datetime(numeric, unit=unit, utc=True, errors="coerce")
        return dates, f"numbers read as '{unit}' since 1970"

    dates = pd.to_datetime(series, utc=True, errors="coerce", format="mixed")
    return dates, "text"


def validate_timestamps(df, label, config):
    col = config.get("timestamp_column")
    if not col:
        record_check(label, "Timestamps", "SKIPPED", "No timestamp_column set in Section 6.")
        return
    if col not in df.columns:
        record_check(label, f"{col}: column exists", "FAIL",
                     "Configured column was not found. Update Section 6.")
        return

    # values read as dates
    dates, how = to_dates(flatten_values(df[col]))
    unreadable = int(dates.isna().sum())
    record_check(label, f"{col}: values read as dates",
                 "PASS" if unreadable == 0 else "FAIL",
                 f"{len(dates):,} values ({how}); {unreadable:,} could not be read.")
    readable = dates.dropna()
    if len(readable) == 0:
        return

    # plausible window
    outside = int(((readable < earliest) | (readable > latest)).sum())
    record_check(label, f"{col}: dates in plausible window",
                 "PASS" if outside == 0 else "FAIL",
                 f"Observed {readable.min()} to {readable.max()}; {outside:,} outside "
                 f"{earliest.date()} to {latest.date()}.")

    # time order: inside each list, or across one user's rows
    user_col     = config.get("user_column")
    holds_lists  = df[col].map(is_list_value).any()
    if holds_lists:
        unsorted = df[col].map(lambda v: is_list_value(v) and not pd.Series(v).is_monotonic_increasing)
        record_check(label, f"{col}: events in time order",
                     "PASS" if not unsorted.any() else "FAIL",
                     f"{int(unsorted.sum()):,} of {len(df):,} sequences are not in time order."
                     + example_user(df, unsorted, user_col))
    elif user_col and user_col in df.columns:
        order = pd.DataFrame({"user": df[user_col], "time": to_dates(df[col])[0]})
        steps = order.groupby("user", sort=False)["time"].diff()
        backwards = steps < pd.Timedelta(0)
        record_check(label, f"{col}: rows in time order per user",
                     "PASS" if not backwards.any() else "WARNING",
                     f"{order.loc[backwards, 'user'].nunique():,} users have rows out of time order."
                     + example_user(df, backwards, user_col))


def validate_ranges(df, label, config):
    for col, (low, high) in (config.get("range_columns") or {}).items():
        if col not in df.columns:
            record_check(label, f"{col}: values in range", "FAIL",
                         "Configured column was not found. Update Section 6.")
            continue
        values = pd.to_numeric(pd.Series(flatten_values(df[col])), errors="coerce")
        bad    = int(values.isna().sum())
        if low is not None:
            bad += int((values < low).sum())
        if high is not None:
            bad += int((values > high).sum())
        if low is not None and high is not None:
            allowed = f"{low} to {high}"
        elif low is not None:
            allowed = f">= {low}"
        else:
            allowed = f"<= {high}"
        observed = f"observed {values.min():g} to {values.max():g}" if values.notna().any() else "no values"
        record_check(label, f"{col}: values in range",
                     "PASS" if bad == 0 else "FAIL",
                     f"Allowed {allowed}; {observed}; {bad:,} values outside or not numbers.")


for label, df, config in [("Encoded", encoded_df, ENCODED), ("Sequenced", sequenced_df, SEQUENCED)]:
    if df is None:
        continue
    run_safely(label, "Timestamps",   validate_timestamps, df, label, config)
    run_safely(label, "Value ranges", validate_ranges, df, label, config)

show_section_results()

## 11. Cross-check: encoded vs sequenced

When both outputs are loaded, they should describe the same data:
- the same set of users,
- for each user, the number of encoded rows (events) equals the number of events in their sequence,
- for each user, the total number of tokens is the same.

Users missing from either side are a FAIL. Event or token counts that differ are a WARNING, because a sequencing step may cap or filter events on purpose. If yours does not, treat it as an error.

This section is skipped when either output is missing, when `MAX_ROWS_PER_OUTPUT` capped the rows, or when the user column is not set.

In [ ]:
start_section("11. Cross-check")


def count_compare(name, encoded_counts, sequenced_counts):
    both     = pd.DataFrame({"encoded": encoded_counts, "sequenced": sequenced_counts}).dropna()
    mismatch = both["encoded"] != both["sequenced"]
    details  = f"{int(mismatch.sum()):,} of {len(both):,} users differ."
    if mismatch.any():
        user = both.index[mismatch][0]
        details += (f" First: {user} (encoded {both.loc[user, 'encoded']:.0f}, "
                    f"sequenced {both.loc[user, 'sequenced']:.0f}).")
    record_check("Both", f"Same {name} per user", "PASS" if not mismatch.any() else "WARNING", details)


def cross_check():
    if encoded_df is None or sequenced_df is None:
        record_check("Both", "Encoded vs sequenced", "SKIPPED", "Needs both outputs.")
        return
    if encoded_capped or sequenced_capped:
        record_check("Both", "Encoded vs sequenced", "SKIPPED", "Rows were capped by MAX_ROWS_PER_OUTPUT.")
        return

    enc_user = ENCODED.get("user_column")
    seq_user = SEQUENCED.get("user_column")
    if enc_user not in encoded_df.columns or seq_user not in sequenced_df.columns:
        record_check("Both", "Encoded vs sequenced", "SKIPPED",
                     "user_column not set, or not found in one of the outputs.")
        return

    # same users
    enc_users = set(encoded_df[enc_user].dropna())
    seq_users = set(sequenced_df[seq_user].dropna())
    only_enc  = sorted(enc_users - seq_users)
    only_seq  = sorted(seq_users - enc_users)
    details   = (f"{len(enc_users):,} users in encoded, {len(seq_users):,} in sequenced; "
                 f"{len(only_enc):,} only in encoded, {len(only_seq):,} only in sequenced.")
    if only_enc or only_seq:
        details += f" First: {(only_enc + only_seq)[0]}."
    record_check("Both", "Same users in both outputs",
                 "PASS" if not only_enc and not only_seq else "FAIL", details)

    # events per user: encoded rows against the events in each sequence
    seq_event_cols = SEQUENCED.get("per_event_columns") or []
    seq_offsets    = SEQUENCED.get("offsets_column")
    if seq_event_cols and seq_event_cols[0] in sequenced_df.columns:
        seq_events = sequenced_df[seq_event_cols[0]].map(list_length)
    elif seq_offsets and seq_offsets in sequenced_df.columns:
        seq_events = sequenced_df[seq_offsets].map(list_length) - 1
    else:
        seq_events = None

    if seq_events is None:
        record_check("Both", "Same events per user", "SKIPPED",
                     "Sequenced output needs per_event_columns or offsets_column.")
    else:
        count_compare("events",
                      encoded_df.groupby(enc_user).size(),
                      seq_events.groupby(sequenced_df[seq_user]).sum())

    # tokens per user
    enc_aligned = ENCODED.get("aligned_columns") or []
    seq_aligned = SEQUENCED.get("aligned_columns") or []
    if not enc_aligned or not seq_aligned:
        record_check("Both", "Same tokens per user", "SKIPPED", "aligned_columns not set for both outputs.")
        return
    enc_tokens = encoded_df[enc_aligned[0]].map(list_length).clip(lower=0)
    seq_tokens = sequenced_df[seq_aligned[0]].map(list_length).clip(lower=0)
    count_compare("tokens",
                  enc_tokens.groupby(encoded_df[enc_user]).sum(),
                  seq_tokens.groupby(sequenced_df[seq_user]).sum())


run_safely("Both", "Encoded vs sequenced", cross_check)
show_section_results()

## 12. Review a few records manually

Automated checks are useful, but always look at a few rows yourself. This cell shows sample rows, then one user's first events laid out token by token. When `VOCAB_FILE` points at `tokenizer.json`, key IDs and categorical or binned values are shown by name.

Look for:
- Token IDs represented in the expected format
- Each event starting with the token you expect (in the reference pipeline, the first token of an event is its `source`)
- Keys and values that belong together (an `amount` key followed by an amount bin, not a merchant name)
- Timestamps that look like real dates, in order

In [ ]:
if encoded_df is not None:
    print("ENCODED SAMPLE")
    display(encoded_df.head(3))

if sequenced_df is not None:
    print("SEQUENCED SAMPLE")
    display(sequenced_df.head(3))


# names for key and value IDs (only when VOCAB_FILE is a tokenizer.json)
key_names   = {token_id: key for key, token_id in vocab_info.get("key_vocab", {}).items()}
value_names = {}
for key, values in vocab_info.get("cat_vocab", {}).items():
    for value, token_id in values.items():
        value_names[token_id] = value
for key, base in vocab_info.get("binner_base", {}).items():
    edges = vocab_info["binners"][key]["edges"]
    value_names[base] = "0"
    for bucket in range(1, len(edges) + 2):
        low  = f"{edges[bucket - 2]:.6g}" if bucket >= 2 else "-inf"
        high = f"{edges[bucket - 1]:.6g}" if bucket - 1 < len(edges) else "inf"
        value_names[base + bucket] = f"[{low}, {high})"


def show_first_events(df, label, config, n_events=5):
    # one user's first events as a token-by-token table
    tokens = config.get("token_columns") or []
    if df is None or len(tokens) < 2 or tokens[0] not in df.columns or tokens[1] not in df.columns:
        return
    user_col = config.get("user_column")
    ts_col   = config.get("timestamp_column")
    off_col  = config.get("offsets_column")

    # events as (time, key IDs, value IDs)
    events = []
    if off_col and off_col in df.columns:
        row     = df.iloc[0]
        offsets = row[off_col]
        times   = row[ts_col] if ts_col in df.columns else [None] * (len(offsets) - 1)
        for e in range(min(n_events, len(offsets) - 1)):
            lo, hi = offsets[e], offsets[e + 1]
            events.append((times[e], row[tokens[0]][lo:hi], row[tokens[1]][lo:hi]))
        user = row[user_col] if user_col in df.columns else "row 0"
    else:
        rows = df
        if user_col in df.columns:
            rows = df[df[user_col] == df[user_col].iloc[0]]
        for _, row in rows.head(n_events).iterrows():
            time = row[ts_col] if ts_col in df.columns else None
            events.append((time, row[tokens[0]], row[tokens[1]]))
        user = df[user_col].iloc[0] if user_col in df.columns else "first rows"

    table = []
    for e, (time, keys, values) in enumerate(events):
        date = to_dates([time])[0].iloc[0] if time is not None else None
        for key_id, value_id in zip(keys, values):
            table.append({"event": e, "time": date,
                          "key_id": int(key_id), "key": key_names.get(int(key_id), ""),
                          "value_id": int(value_id), "value": value_names.get(int(value_id), "")})

    print(f"\n{label}: first {len(events)} events of user {user}")
    with pd.option_context("display.max_rows", None):
        display(pd.DataFrame(table))


# a display problem here must not stop Run All before the report
for label, df, config in [("ENCODED", encoded_df, ENCODED), ("SEQUENCED", sequenced_df, SEQUENCED)]:
    try:
        show_first_events(df, label, config)
    except Exception as exc:
        print(f"{label}: could not lay out the first events ({type(exc).__name__}: {exc})")

## 13. Final validation report

**How to read the result**
- **PASS** – the check ran and passed.
- **WARNING** – something needs review. It may be valid for your data design, for example duplicate rows or a sequencing step that caps events on purpose.
- **FAIL** – the output breaks a rule the model relies on and should be investigated.
- **ERROR** – the check itself crashed, usually because a column holds a different type than expected. Read the details, then fix the setting or the data.
- **SKIPPED** – the check was not configured or the output was not provided. It does not change the overall result.

Problems are listed first.

In [ ]:
report = pd.DataFrame(list(results.values()))

if report.empty:
    print("No checks were executed.")
else:
    # problems first; a stable sort keeps the run order inside each status
    status_order = {"FAIL": 0, "ERROR": 1, "WARNING": 2, "PASS": 3, "SKIPPED": 4}
    report["order"] = report["status"].map(status_order)
    report = (report.sort_values("order", kind="stable")
                    .drop(columns="order")
                    .reset_index(drop=True))

    print("SUMMARY")
    display(report["status"].value_counts().rename_axis("status").to_frame("count"))

    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(report)

    if report["status"].isin(["FAIL", "ERROR"]).any():
        print("\nOVERALL RESULT: FAIL - investigate the failed checks and errors.")
    elif (report["status"] == "WARNING").any():
        print("\nOVERALL RESULT: REVIEW - check warnings before accepting the output.")
    else:
        print("\nOVERALL RESULT: PASS - all executed checks passed.")

# Optional: save the report next to the notebook's current working directory.
# report.to_csv("data_pipeline_validation_report.csv", index=False)

## 14. What to do if a check fails

1. Read the failed check and its details in the report. Most details name the first user or row with the problem.
2. Go back to **Section 5** and confirm the actual schema and column names.
3. Check whether the relevant column is set correctly in **Section 6**.
4. Look at the sample rows and the token-by-token view in **Section 12**.
5. If the data itself is incorrect, revisit the corresponding pipeline stage and regenerate the output.
6. Run the notebook again (**Run All**) after fixing the issue.

**Common failures and their usual cause**

| Failing check | Usual cause |
|---|---|
| Token lists do not line up | Key and value lists built in separate loops, or one list cut without the others |
| Last offset is not the token count | Offsets computed before tokens were cut or capped |
| Per-event lists do not line up | Timestamps or calendar fields taken from a different set of events than the tokens |
| Dates outside the plausible window | Timestamp unit mixed up (seconds vs milliseconds vs microseconds) |
| Events not in time order | Events grouped per user without sorting by time first |
| Token ID above the vocabulary | Tokenizer from a different run, or an ID offset added twice |
| High `[UNK]` share | Tokenizer fitted on different data, or keys renamed after fitting |
| `[PAD]` / `[MASK]` tokens present | Padding or masking done in the data pipeline instead of at batching / training time |
| User appears in more than one row | Output written in chunks that split one user's history |
| Users missing between outputs | A filter or join in the sequencing step dropped users |

**Remember:** This notebook validates the generated output. It does not modify your Parquet files or retrain the foundation model.